# van der Pauw conductivity: from raw files to σ(T) and Ea (DEMO)

**The data are fictitious** (`demo_data/make_demo_data.py`), but messy the way real ones are:

* the vdP program writes **four resistances** per point (R12,34 and R23,41, each with + and − current),
  semicolons, **decimal comma**, **local time**, a few header lines, and it was **restarted once** (two files);
  some points are broken (contact glitch, overload);
* the temperature comes from a **separate furnace log**, in UTC, every 10 s, from a PC whose **clock is off**.

What the notebook does:

1. the run and sample from AMBER / `AMBER_run.json` (thickness from the run parameters);
2. reads and joins both vdP files, converts local time to UTC, removes broken points;
3. reverses-current average and the thermovoltage; checks contact symmetry;
4. solves the **van der Pauw equation** for the sheet resistance → ρ and σ;
5. finds the **clock offset** between the two PCs and matches each point with the sample temperature;
6. keeps only points on **stable temperature plateaus** and averages them;
7. **Arrhenius** (small polaron, ln σT vs 1/T) → activation energy;
8. a result table for the experiment.

Everything you might need to change is in the first code cell.

## 0. Setup

In [ ]:
%pip install -q git+https://github.com/dfme-lab/amber-notebooks

In [ ]:
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from amber_client import Amber, AmberError, load_run_json

RUN = "ST99-26r001"                     # run code (this demo run is not in AMBER)
AMBER_URL = "https://test.next.dfme.eu"
FOLDER = Path("data") / RUN
RESULTS = Path("results") / RUN
RESULTS.mkdir(parents=True, exist_ok=True)

LOCAL_TZ = "Europe/Warsaw"   # time zone of the vdP PC
MAX_GAP_S = 60               # a vdP point needs a temperature reading within this time
PLATEAU_RATE = 0.2           # K/min: slower than this counts as a stable temperature
PLATEAU_MIN = 15             # min: a plateau must last at least this long
SKIP_MIN = 10                # min: drop the first minutes of each plateau (sample still settling)

## 1. The run and the sample

From AMBER with your key if the run is there; the data folder's `AMBER_run.json` in any case.

In [ ]:
DEMO = "https://raw.githubusercontent.com/dfme-lab/amber-notebooks/main/stations/vdp/demo_data/" + RUN + "/"
if not (FOLDER / "AMBER_run.json").exists():
    FOLDER.mkdir(parents=True, exist_ok=True)
    for name in urllib.request.urlopen(DEMO + "MANIFEST.txt").read().decode().split():
        urllib.request.urlretrieve(DEMO + name, FOLDER / name)
    print("Demo folder downloaded")

try:
    run = Amber(AMBER_URL).run(RUN)
    print("AMBER:", run["code"], "|", run.get("title"), "|", run.get("status"))
except AmberError as e:
    print("Not read from AMBER:", e)

info = load_run_json(FOLDER)
r = info["run"]
print(r["code"], "·", r["title"])
for s in info["samples"]:
    print("sample:", s["code"], "·", s.get("title") or "")
THICK_CM = float(r["params"]["thickness_mm"]) / 10
print("thickness:", THICK_CM * 10, "mm")

## 2. Read the vdP files

All `vdP_*.txt` files are joined. Header lines (`#`) are skipped, decimal comma and local time are handled, broken points are dropped and counted.

In [ ]:
COLS = {"R12_34_plus [Ohm]": "RA_p", "R12_34_minus [Ohm]": "RA_m",
        "R23_41_plus [Ohm]": "RB_p", "R23_41_minus [Ohm]": "RB_m"}
parts = []
for f in sorted(FOLDER.glob("vdP_*.txt")):
    d = pd.read_csv(f, sep=";", decimal=",", comment="#", na_values=["NaN", "OVER"])
    d["file"] = f.name
    parts.append(d)
vdp = pd.concat(parts, ignore_index=True).rename(columns=COLS)
local = pd.to_datetime(vdp["Data"] + " " + vdp["Czas"], format="%d.%m.%Y %H:%M:%S")
vdp["time"] = local.dt.tz_localize(LOCAL_TZ).dt.tz_convert("UTC")
n0 = len(vdp)
bad_status = vdp["Status"].ne("OK") | vdp[list(COLS.values())].isna().any(axis=1)
vdp = vdp[~bad_status].sort_values("time").reset_index(drop=True)
print(f"{n0} points in {len(parts)} files; {bad_status.sum()} broken (overload / empty) removed")
print("gaps longer than 5 min:")
gap = vdp["time"].diff()
print(vdp.loc[gap > pd.Timedelta(minutes=5), ["time", "file"]].assign(gap=gap[gap > pd.Timedelta(minutes=5)]))

## 3. Current reversal, thermovoltage, contact symmetry

* R_A = (R12,34⁺ + R12,34⁻)/2, R_B the same: averaging the two current directions removes the **thermovoltage**,
  V_th = (R⁺ − R⁻)·I/2.
* A **glitch** shows as + and − readings that disagree far more than usual: such points are removed.
* R_A/R_B tells how symmetric the contacts and the sample are (van der Pauw works for any ratio, but a
  ratio that drifts during the run means a contact is changing).

In [ ]:
I = vdp["I [mA]"] / 1000
vdp["RA"] = (vdp["RA_p"] + vdp["RA_m"]) / 2
vdp["RB"] = (vdp["RB_p"] + vdp["RB_m"]) / 2
vdp["Vth_A_uV"] = (vdp["RA_p"] - vdp["RA_m"]) / 2 * I * 1e6
vdp["Vth_B_uV"] = (vdp["RB_p"] - vdp["RB_m"]) / 2 * I * 1e6
mism = np.maximum(abs(vdp["RA_p"] - vdp["RA_m"]) / vdp["RA"], abs(vdp["RB_p"] - vdp["RB_m"]) / vdp["RB"])
glitch = mism > 5 * mism.median() + 0.02
print(f"{glitch.sum()} glitch points removed (+/- readings disagree)")
vdp = vdp[~glitch].reset_index(drop=True)
vdp["ratio"] = vdp["RA"] / vdp["RB"]
print("R_A/R_B: median", round(vdp["ratio"].median(), 3), " min", round(vdp["ratio"].min(), 3),
      " max", round(vdp["ratio"].max(), 3))

## 4. van der Pauw equation

exp(−πR_A/R_s) + exp(−πR_B/R_s) = 1 is solved for the sheet resistance R_s (Newton). Then ρ = R_s·t and σ = 1/ρ.

In [ ]:
def sheet_resistance(ra, rb, iterations=50):
    ra, rb = np.asarray(ra, float), np.asarray(rb, float)
    rs = np.pi * (ra + rb) / (2 * np.log(2))          # exact for ra == rb
    for _ in range(iterations):
        ea, eb = np.exp(-np.pi * ra / rs), np.exp(-np.pi * rb / rs)
        f = ea + eb - 1
        df = (np.pi / rs**2) * (ra * ea + rb * eb)
        rs = rs - f / df
    return rs

vdp["Rs_ohm"] = sheet_resistance(vdp["RA"], vdp["RB"])
vdp["rho_ohm_cm"] = vdp["Rs_ohm"] * THICK_CM
vdp["sigma_S_cm"] = 1 / vdp["rho_ohm_cm"]
check = np.exp(-np.pi * vdp["RA"] / vdp["Rs_ohm"]) + np.exp(-np.pi * vdp["RB"] / vdp["Rs_ohm"])
print("equation residual (should be ~0):", float(abs(check - 1).max()))

## 5. Clock offset and temperature

The two PCs do not agree on the time. The offset is found by trying shifts of the furnace clock from −10 to
+10 min: the right one gives the smoothest ln(σT) against 1/T (on ramps, a wrong time gives a wrong temperature).
Then each vdP point gets the temperature interpolated at its time.

In [ ]:
temp = pd.read_csv(FOLDER / "furnace_log.csv")
temp["time"] = pd.to_datetime(temp["timestamp"], utc=True)

def matched(offset_s):
    t = temp["time"] - pd.Timedelta(seconds=offset_s)
    x = (t - t.iloc[0]).dt.total_seconds().to_numpy()
    xv = (vdp["time"] - t.iloc[0]).dt.total_seconds().to_numpy()
    T = np.interp(xv, x, temp["pv_C"].to_numpy(), left=np.nan, right=np.nan)
    near = np.abs(xv[:, None] - x[None, ::30]).min(axis=1) <= MAX_GAP_S * 30   # inside the log
    T[~near] = np.nan
    return T

after_peak = (vdp["time"] > vdp["time"].iloc[int(np.argmax(vdp["sigma_S_cm"]))]).to_numpy()

def scatter(offset_s):  # only after the hottest point: heating can differ from cooling
    T = matched(offset_s); ok = ~np.isnan(T) & after_peak
    x, y = 1 / (T[ok] + 273.15), np.log(vdp["sigma_S_cm"].to_numpy()[ok] * (T[ok] + 273.15))
    return np.std(y - np.polyval(np.polyfit(x, y, 3), x))

shifts = np.arange(-600, 601, 2)
s = [scatter(o) for o in shifts]
OFFSET_S = int(shifts[int(np.argmin(s))])
print(f"furnace clock is {OFFSET_S:+d} s off (positive = fast)")
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(shifts / 60, s); ax.axvline(OFFSET_S / 60, color="r", ls=":")
ax.set_xlabel("assumed furnace clock error (min)"); ax.set_ylabel("scatter of ln σT"); plt.show()

vdp["T_C"] = matched(OFFSET_S)

## 6. The whole run, and the stable plateaus

Only points where the temperature changed less than `PLATEAU_RATE` for at least `PLATEAU_MIN` are used, without the first `SKIP_MIN`.

In [ ]:
t_temp = temp["time"] - pd.Timedelta(seconds=OFFSET_S)
rate = pd.Series(temp["pv_C"].to_numpy(), index=t_temp).rolling("5min").apply(
    lambda v: (v[-1] - v[0]) / 5 if len(v) > 1 else np.nan, raw=True)
stable = rate.abs() < PLATEAU_RATE
block = (stable != stable.shift()).cumsum()
plateaus = []
for _, g in stable.groupby(block):
    if g.iloc[0] and (g.index[-1] - g.index[0]) >= pd.Timedelta(minutes=PLATEAU_MIN):
        plateaus.append((g.index[0] + pd.Timedelta(minutes=SKIP_MIN), g.index[-1]))

vdp["plateau"] = -1
for k, (a, b) in enumerate(plateaus):
    vdp.loc[(vdp["time"] >= a) & (vdp["time"] <= b), "plateau"] = k
peak = vdp.loc[vdp["T_C"].idxmax(), "time"]
vdp["branch"] = np.where(vdp["time"] < peak, "heating", "cooling")

t0 = t_temp.iloc[0]
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot((t_temp - t0).dt.total_seconds() / 3600, temp["pv_C"], lw=1, label="furnace T")
for a, b in plateaus:
    ax.axvspan((a - t0).total_seconds() / 3600, (b - t0).total_seconds() / 3600, color="g", alpha=0.15)
ax.set_xlabel("time (h)"); ax.set_ylabel("T (°C)")
ax2 = ax.twinx()
h = (vdp["time"] - t0).dt.total_seconds() / 3600
ax2.semilogy(h, vdp["sigma_S_cm"], ".", ms=3, color="tab:red", label="σ")
ax2.set_ylabel("σ (S/cm)")
ax.set_title(f"{RUN}: temperature, σ and the stable plateaus (green)")
fig.tight_layout(); fig.savefig(RESULTS / "overview.png", dpi=150); plt.show()

plat = (vdp[vdp["plateau"] >= 0].groupby("plateau")
        .agg(T_C=("T_C", "mean"), sigma_S_cm=("sigma_S_cm", "mean"), sigma_sd=("sigma_S_cm", "std"),
             n=("sigma_S_cm", "size"), ratio=("ratio", "mean"), Vth_uV=("Vth_A_uV", "mean"),
             branch=("branch", "first"))
        .round({"T_C": 1, "sigma_S_cm": 3, "sigma_sd": 3, "ratio": 3, "Vth_uV": 1}))
plat

## 7. Arrhenius (small polaron)

σT = A·exp(−Ea/kT), so ln(σT) against 1000/T is a line with slope −Ea/k. Fitted on the cooling plateaus; all points are shown in grey.

In [ ]:
kB = 8.617e-5
cool = plat[plat["branch"] == "cooling"]
if len(cool) >= 2:
    x = 1000 / (cool["T_C"] + 273.15); y = np.log(cool["sigma_S_cm"] * (cool["T_C"] + 273.15))
    (k, b), cov = np.polyfit(x, y, 1, cov=True)
    EA = -k * 1000 * kB; EA_ERR = np.sqrt(cov[0, 0]) * 1000 * kB
    fig, ax = plt.subplots(figsize=(6, 4))
    ok = vdp["T_C"].notna()
    xa = 1000 / (vdp.loc[ok, "T_C"] + 273.15)
    ax.plot(xa, np.log(vdp.loc[ok, "sigma_S_cm"] * (vdp.loc[ok, "T_C"] + 273.15)), ".", color="0.8", ms=3, label="all points")
    ax.plot(x, y, "o", label="cooling plateaus")
    ax.plot(x, np.polyval([k, b], x), "-", label=f"Ea = {EA:.3f} ± {EA_ERR:.4f} eV")
    ax.set_xlabel("1000/T (1/K)"); ax.set_ylabel("ln(σT)  (σ in S/cm, T in K)"); ax.legend()
    fig.tight_layout(); fig.savefig(RESULTS / "arrhenius.png", dpi=150); plt.show()
else:
    EA = EA_ERR = float("nan")
    print("Fewer than two stable plateaus on cooling: no Arrhenius fit.")

## 8. Result for the experiment

Copy into the experiment in AMBER (nothing is written back automatically). Figures and the table are in `results/`.

In [ ]:
at = lambda T: np.exp(np.polyval([k, b], 1000 / (T + 273.15))) / (T + 273.15) if len(cool) >= 2 else float("nan")
table = cool[["T_C", "sigma_S_cm", "sigma_sd", "n", "ratio"]]
table.to_csv(RESULTS / "plateaus.csv", index=False)
summary = (f"Run {RUN}, sample {', '.join(s['code'] for s in info['samples'])}, thickness {THICK_CM*10:.2f} mm, "
           f"{len(vdp)} good vdP points ({n0 - len(vdp)} removed), furnace clock offset {OFFSET_S:+d} s.\n"
           f"Ea (cooling, {len(cool)} plateaus) = {EA:.3f} ± {EA_ERR:.4f} eV; "
           f"σ(800 °C) = {at(800):.1f} S/cm, σ(600 °C) = {at(600):.1f} S/cm (from the fit).\n\n"
           + table.to_string(index=False))
(RESULTS / "summary.txt").write_text(summary, encoding="utf-8")
print(summary)